In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import plotly.express as px
output = "output_plot"
df_raw = pd.read_csv("Mall_Customers.csv").dropna()
df_raw["GenderNo"] = np.where(df_raw["Gender"] == "Male", 0, 1)  # 0/1 instead of 1/3 - cleaner, no reason to skip 2
df = df_raw[["GenderNo", "Annual_Income", "Spending_Score"]].copy()
scaler = StandardScaler()
dfnew = scaler.fit_transform(df)
#  Pick k via silhouette score, with a plot so you can eyeball it instead of trusting argmax blindly 
scores = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(dfnew)
    scores[k] = silhouette_score(dfnew, km.labels_)
plt.figure(figsize=(6, 4))
plt.plot(list(scores.keys()), list(scores.values()), marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette score")
plt.title("KMeans: silhouette score vs k")
plt.savefig(f"{output}/silhouette_scores.png")
plt.close()
best_k = max(scores, key=scores.get)
km = KMeans(n_clusters=best_k, n_init=10, random_state=42)
labels = km.fit_predict(dfnew)
df["Label_Kmeans"] = labels
spend_mean = df["Spending_Score"].mean()
income_mean = df["Annual_Income"].mean()
def label(row):
    hi_spend = row["Spending_Score"] > spend_mean
    low_spend = row["Spending_Score"] <= spend_mean  # was "<", meaning a value exactly at the mean matched neither -> fell through to None
    hi_salary = row["Annual_Income"] > income_mean
    low_salary = row["Annual_Income"] <= income_mean
    male = row["GenderNo"] == 0
    female = row["GenderNo"] == 1
    gender_str = "Male" if male else "Female"
    spend_str = "High Spend" if hi_spend else "Low Spend"
    income_str = "High Income" if hi_salary else "Low Income"
    return f"{gender_str} With {spend_str} and {income_str}"
df["About"] = df.apply(label, axis=1)
# DBSCAN: pick eps from the k-distance elbow instead of guessing 0.4/0.5 
k = 5  # matches min_samples below
neighbors = NearestNeighbors(n_neighbors=k).fit(dfnew)
distances, _ = neighbors.kneighbors(dfnew)
k_distances = np.sort(distances[:, -1])
plt.figure(figsize=(6, 4))
plt.plot(k_distances)
plt.xlabel("Points sorted by distance")
plt.ylabel(f"{k}-th nearest neighbor distance")
plt.title("DBSCAN eps elbow plot")
plt.savefig(f"{output}/dbscan_elbow.png")
plt.close()
db = DBSCAN(eps=0.4, min_samples=5)  # read the elbow plot and adjust eps if needed
df["Label_DBSCAN"] = db.fit_predict(dfnew)
df["Age"] = df_raw["Age"]
df["Gender"] = df_raw["Gender"]
df["CustomerID"] = df_raw["CustomerID"]
df.to_csv("new.csv", index=False)  # index=False so we don't get an extra "Unnamed: 0" column on reload
def recmendation_by_text(text):
    if not isinstance(text, str) or not text.strip():
        print("Please give a valid, non-empty string")
        return []
    text = text.lower()
    mask = pd.Series(True, index=df.index)  # start with everything, then narrow down
    matched_any = False
    if "male" in text and "female" not in text:
        mask &= df["Gender"] == "Male"
        matched_any = True
    elif "female" in text:
        mask &= df["Gender"] == "Female"
        matched_any = True
    if "income" in text:
        if "high" in text:
            mask &= df["Annual_Income"] > income_mean
            matched_any = True
        elif "low" in text:
            mask &= df["Annual_Income"] < income_mean
            matched_any = True
    if "spend" in text:
        if "high" in text:
            mask &= df["Spending_Score"] > spend_mean
            matched_any = True
        elif "low" in text:
            mask &= df["Spending_Score"] < spend_mean
            matched_any = True
    if not matched_any:
        print("No recognized keywords (try: male/female, high/low income, high/low spend)")
        return []
    return df.loc[mask, "CustomerID"].tolist()
def recomendation_by_person(customer_id):
    about_values = df.loc[df["CustomerID"] == customer_id, "About"]
    if about_values.empty:
        print(f"No record found for CustomerID {customer_id}")
        return []
    about_value = about_values.iloc[0]
    matching_ids = df.loc[df["About"] == about_value, "CustomerID"]
    return matching_ids.tolist()


In [ ]:
def plot_2d_matplotlib(df, label_col, fname):
    plt.figure(figsize=(8, 6))
    labels = df[label_col].unique()
    for lab in sorted(labels, key=str):
        sub = df[df[label_col] == lab]
        plt.scatter(sub["Annual_Income"], sub["Spending_Score"], label=str(lab), s=40, alpha=0.75)
    plt.xlabel("Annual Income")
    plt.ylabel("Spending Score")
    plt.title(f"Customer Clusters ({label_col})")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()
def plot_3d_matplotlib(df, label_col, fname):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    labels = sorted(df[label_col].unique(), key=str)
    for lab in labels:
        sub = df[df[label_col] == lab]
        ax.scatter(sub["Age"], sub["Annual_Income"], sub["Spending_Score"], label=str(lab), s=30, alpha=0.75)
    ax.set_xlabel("Age")
    ax.set_ylabel("Annual Income (k$)")
    ax.set_zlabel("Spending Score")
    ax.set_title(f"3D Customer Clusters ({label_col})")
    ax.legend(loc="best", fontsize=7)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()
plot_2d_matplotlib(df,"Label_Kmeans", f"{output}/kmeans_2d.png")
plot_3d_matplotlib(df,"Label_Kmeans", f"{output}/kmeans_3d.png")
plot_2d_matplotlib(df,"Label_DBSCAN", f"{output}/DBSCAN_2d.png")
plot_3d_matplotlib(df,"Label_DBSCAN", f"{output}/DBSCAN_3d.png")